In [ ]:
mapfile = "../../test/makemap/map.ubi"
energy = 37.0 # keV
wavelength = 12.3984 / energy
symmetry = 2    # space group number
# ID11 Huber tower:
diffrymax = 11  # degrees
rxlim = 15
rylim = 20

In [ ]:
from ImageD11 import grain, unitcell, parameters
import numpy as np

from scipy.spatial.transform import Rotation

def tilt_grain( gvec ):
    """Solves to put gvec along rotation axis"""
    phix = np.arctan2( gvec[1], gvec[2] )
    Rx = Rotation.from_euler( 'XYZ', (phix,0,0) )
    grot = Rx.apply( gvec )
    phiy = np.arctan2( -grot[0], grot[2] )
    return np.degrees( phix ), np.degrees( phiy )

In [ ]:
# read the grains and determine the max tilt of the base to give a dsmax
grains = grain.read_grain_file(mapfile)
sinthlmax = np.sin( np.radians(diffrymax) ) / wavelength
dsmax = 2*sinthlmax
print(dsmax)

In [ ]:
# generate reflections, assuming all grains have the same symmetry
g = grains[0]
uc = unitcell.unitcell( g.unitcell, symmetry )
pks = uc.gethkls( dsmax )
hkls = np.array( [p[1] for p in pks ] , int)

In [ ]:
# sort by size if you want to
# order = np.argsort([float(g.intensity_info.split()[2]) for g in grains])
print("Warning: needs to be re-tested, last used in 2022 !!\n")
for i in range(len(grains)):
    g = grains[i]
    print("\n".join( ["Grain: "+str(i),"UBI: "+str(g.ubi), 'Rod: '+str(g.Rod), "Cell: "+str(g.unitcell) ] ) )
    print('   h    k    l    d*       theta      rx       ry ')
    for hkl in hkls:
        gvec = np.dot( g.UB, hkl )
        ds = np.sqrt((gvec**2).sum())
        rx, ry = tilt_grain( gvec )
        if abs(rx)<rxlim and abs(ry)<rylim:
            print(('%4d '*3)%tuple(hkl), ('% 9.5f '*4)%(ds, np.degrees(np.arcsin( ds * wavelength / 2)), rx, ry))
    print()